# Optimizacion de Hiperparametros v2 — GridSearchCV Anti-Overfitting

**Cambios respecto a v1:**

| # | Cambio | Antes | Ahora |
|:-:|:-------|:------|:------|
| 1 | Metrica de optimizacion | `scoring='accuracy'` | `scoring='f1_macro'` |
| 2 | Balanceo | Excluyente (SMOTE o class_weight) | Hibrido (SMOTE + class_weight simultaneos) |
| 3 | SMOTE controlado | `sampling_strategy='auto'` (genera miles) | `sampling_strategy={2: 300}` (controlado) |
| 4 | Feature Selection | Ninguna (45 variables) | `SelectKBest(f_classif, k=[10,15,20,25])` |

> **Motivacion**: El accuracy global enmascara la ceguera ante la Clase 2 (0% F1).
> Al optimizar con F1-macro, penalizamos configuraciones que ignoran la clase minoritaria.

## 1. Importaciones y configuracion global

In [1]:
import time
import numpy as np
import pandas as pd

# Pipeline de imblearn (soporta SMOTE dentro de CV sin data leakage)
from imblearn.pipeline import Pipeline as ImbPipeline

from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif

# Modelos
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Metricas
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Configuracion global
SEED: int = 42
CV_FOLDS: int = 5
N_JOBS: int = -1
SCORING: str = 'f1_macro'  # v2: antes 'accuracy'

# SMOTE controlado: solo Clase 2 sube a 300 muestras (desde 19)
SMOTE_STRATEGY = {2: 300}

np.random.seed(SEED)

cv_strategy = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

print('Imports OK')
print(f'Scoring: {SCORING}')
print(f'SMOTE strategy: {SMOTE_STRATEGY}')

Imports OK
Scoring: f1_macro
SMOTE strategy: {2: 300}


## 2. Carga de datos

In [2]:
train_df = pd.read_csv('data/training.csv', index_col=0)
test_df  = pd.read_csv('data/test.csv', index_col=0)

# Variables a eliminar (gemelas con |correlacion| > 0.99)
cols_to_drop = ['bpsmt', 'csuhz', 'gwrec', 'glhls', 'bqwyz']
TARGET = 'class'

X_train = train_df.drop(columns=[TARGET] + cols_to_drop)
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET] + cols_to_drop)

print(f'Features tras seleccion: {X_train.shape[1]} (eliminadas {len(cols_to_drop)})')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'\nDistribucion de clases (train):')
print(y_train.value_counts().sort_index())

Features tras seleccion: 45 (eliminadas 5)
X_train: (3000, 45), X_test: (2000, 45)

Distribucion de clases (train):
class
0    2060
1     921
2      19
Name: count, dtype: int64


## 3. Definicion de Pipelines y Rejillas de Hiperparametros

**Estrategia hibrida v2:** Todos los modelos usan `ImbPipeline` con:
1. `StandardScaler` — estandarizacion
2. `SMOTE(sampling_strategy={2: 300})` — sobremuestreo controlado de Clase 2
3. `SelectKBest(f_classif)` — seleccion de variables (k en rejilla)
4. Clasificador — con `class_weight='balanced'` cuando lo soporte

Esto combina SMOTE + class_weight de forma simultanea para RF, SVM y Bagging.

### 3.1 Random Forest (Hibrido: SMOTE + `class_weight='balanced'`)

In [3]:
pipeline_rf = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=SEED)),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', RandomForestClassifier(
        class_weight='balanced',
        random_state=SEED
    ))
])

param_grid_rf = {
    'feature_selection__k': [10, 15, 20, 25],
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10, 20],
    'classifier__max_features': ['sqrt', 'log2'],
}

n_rf = 4 * 3 * 3 * 2
print(f'Pipeline RF: {list(pipeline_rf.named_steps.keys())}')
print(f'Combinaciones: {n_rf}')

Pipeline RF: ['scaler', 'smote', 'feature_selection', 'classifier']
Combinaciones: 72


### 3.2 Gradient Boosting (SMOTE controlado + SelectKBest)

In [4]:
pipeline_gb = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=SEED)),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', GradientBoostingClassifier(random_state=SEED))
])

param_grid_gb = {
    'feature_selection__k': [10, 15, 20, 25],
    'classifier__n_estimators': [100, 150, 200],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [3, 4, 5],
}

n_gb = 4 * 3 * 3 * 3
print(f'Pipeline GB: {list(pipeline_gb.named_steps.keys())}')
print(f'Combinaciones: {n_gb}')

Pipeline GB: ['scaler', 'smote', 'feature_selection', 'classifier']
Combinaciones: 108


### 3.3 AdaBoost (SMOTE controlado + SelectKBest)

In [5]:
pipeline_ada = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=SEED)),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        random_state=SEED
    ))
])

param_grid_ada = {
    'feature_selection__k': [10, 15, 20, 25],
    'classifier__n_estimators': [50, 100, 200],
    'classifier__learning_rate': [0.05, 0.1, 1.0],
}

n_ada = 4 * 3 * 3
print(f'Pipeline AdaBoost: {list(pipeline_ada.named_steps.keys())}')
print(f'Combinaciones: {n_ada}')

Pipeline AdaBoost: ['scaler', 'smote', 'feature_selection', 'classifier']
Combinaciones: 36


### 3.4 SVM con Kernel RBF (Hibrido: SMOTE + `class_weight='balanced'`)

In [6]:
pipeline_svm = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=SEED)),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', SVC(
        kernel='rbf',
        class_weight='balanced',
        random_state=SEED
    ))
])

param_grid_svm = {
    'feature_selection__k': [10, 15, 20, 25],
    'classifier__C': [1, 10, 100],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1],
}

n_svm = 4 * 3 * 4
print(f'Pipeline SVM: {list(pipeline_svm.named_steps.keys())}')
print(f'Combinaciones: {n_svm}')

Pipeline SVM: ['scaler', 'smote', 'feature_selection', 'classifier']
Combinaciones: 48


### 3.5 Bagging (Hibrido: SMOTE + `class_weight='balanced'` via estimador base)

In [7]:
pipeline_bag = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(sampling_strategy=SMOTE_STRATEGY, random_state=SEED)),
    ('feature_selection', SelectKBest(score_func=f_classif)),
    ('classifier', BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight='balanced'),
        random_state=SEED
    ))
])

param_grid_bag = {
    'feature_selection__k': [10, 15, 20, 25],
    'classifier__n_estimators': [100, 150, 200],
    'classifier__max_samples': [0.7, 0.8, 1.0],
}

n_bag = 4 * 3 * 3
print(f'Pipeline Bagging: {list(pipeline_bag.named_steps.keys())}')
print(f'Combinaciones: {n_bag}')

Pipeline Bagging: ['scaler', 'smote', 'feature_selection', 'classifier']
Combinaciones: 36


## 4. Ejecucion de GridSearchCV

**Configuracion comun v2:**
- `cv=5` (StratifiedKFold con shuffle)
- `scoring='f1_macro'` (antes: `accuracy`)
- `n_jobs=-1` (paralelizacion maxima)
- `refit=True`

In [8]:
models_config = [
    {
        'name': 'Random Forest',
        'pipeline': pipeline_rf,
        'param_grid': param_grid_rf,
        'strategy': 'Hibrido: SMOTE({2:300}) + class_weight',
    },
    {
        'name': 'Gradient Boosting',
        'pipeline': pipeline_gb,
        'param_grid': param_grid_gb,
        'strategy': 'SMOTE({2:300}) + SelectKBest',
    },
    {
        'name': 'AdaBoost',
        'pipeline': pipeline_ada,
        'param_grid': param_grid_ada,
        'strategy': 'SMOTE({2:300}) + SelectKBest',
    },
    {
        'name': 'SVM (RBF)',
        'pipeline': pipeline_svm,
        'param_grid': param_grid_svm,
        'strategy': 'Hibrido: SMOTE({2:300}) + class_weight',
    },
    {
        'name': 'Bagging',
        'pipeline': pipeline_bag,
        'param_grid': param_grid_bag,
        'strategy': 'Hibrido: SMOTE({2:300}) + class_weight (base)',
    },
]

print(f'Modelos registrados: {[m["name"] for m in models_config]}')
total_combos = n_rf + n_gb + n_ada + n_svm + n_bag
print(f'Total combinaciones (todos los modelos): {total_combos}')
print(f'Total fits (x {CV_FOLDS} folds): {total_combos * CV_FOLDS}')

Modelos registrados: ['Random Forest', 'Gradient Boosting', 'AdaBoost', 'SVM (RBF)', 'Bagging']
Total combinaciones (todos los modelos): 300
Total fits (x 5 folds): 1500


In [ ]:
grid_results = []

for config in models_config:
    name = config['name']
    pipeline = config['pipeline']
    param_grid = config['param_grid']
    strategy = config['strategy']

    n_combinations = int(np.prod([len(v) for v in param_grid.values()]))

    print(f"\n{'='*70}")
    print(f'  MODELO: {name}')
    print(f'  Estrategia de balanceo: {strategy}')
    print(f'  Combinaciones: {n_combinations} x {CV_FOLDS} folds = {n_combinations * CV_FOLDS} fits')
    print(f"{'='*70}")

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring=SCORING,
        n_jobs=N_JOBS,
        verbose=1,
        refit=True,
    )

    start_time = time.time()
    grid_search.fit(X_train, y_train)
    elapsed = time.time() - start_time

    print(f'\n  Mejor score ({SCORING}): {grid_search.best_score_:.4f}')
    print(f'  Tiempo total: {elapsed:.1f}s')
    print(f'  Mejores parametros:')
    for param, value in grid_search.best_params_.items():
        clean_param = param.replace('classifier__', '').replace('feature_selection__', '')
        print(f'     - {clean_param}: {value}')

    grid_results.append({
        'Modelo': name,
        'Estrategia': strategy,
        f'Mejor {SCORING}': grid_search.best_score_,
        'Mejores Parametros': grid_search.best_params_,
        'Tiempo (s)': round(elapsed, 1),
        'grid_search': grid_search,
    })


  MODELO: Random Forest
  Estrategia de balanceo: Hibrido: SMOTE({2:300}) + class_weight
  Combinaciones: 72 x 5 folds = 360 fits
Fitting 5 folds for each of 72 candidates, totalling 360 fits

  Mejor score (f1_macro): 0.4324
  Tiempo total: 168.1s
  Mejores parametros:
     - max_depth: 20
     - max_features: sqrt
     - n_estimators: 300
     - k: 25

  MODELO: Gradient Boosting
  Estrategia de balanceo: SMOTE({2:300}) + SelectKBest
  Combinaciones: 108 x 5 folds = 540 fits
Fitting 5 folds for each of 108 candidates, totalling 540 fits


## 5. Resumen comparativo

In [ ]:
summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'grid_search'}
    for r in grid_results
]).sort_values(f'Mejor {SCORING}', ascending=False).reset_index(drop=True)

display(summary_df[['Modelo', 'Estrategia', f'Mejor {SCORING}', 'Tiempo (s)']])

best = summary_df.iloc[0]
print(f"\nMejor modelo global: {best['Modelo']} "
      f"con {SCORING} = {best[f'Mejor {SCORING}']:.4f}")

## 6. Detalle de mejores parametros por modelo

In [ ]:
for result in grid_results:
    print(f"\n{'='*50}")
    print(f"  {result['Modelo']} ({result['Estrategia']})")
    print(f"  Score: {result[f'Mejor {SCORING}']:.4f}")
    print(f"  Parametros optimos:")
    for param, value in result['Mejores Parametros'].items():
        clean_param = param.replace('classifier__', '').replace('feature_selection__', '')
        print(f"     {clean_param}: {value}")

## 7. Control de Overfitting: Brecha Train vs CV

Verificamos la diferencia entre las metricas sobre el propio training (memorizado)
y las metricas reales del CV (generalizacion).

In [ ]:
print(f"{'Modelo':<22} {'Train Acc':>10} {'Train F1m':>10} {'CV F1m':>10} {'Brecha':>10}")
print('=' * 65)

for result in grid_results:
    name = result['Modelo']
    gs = result['grid_search']
    best_pipe = gs.best_estimator_

    train_preds = best_pipe.predict(X_train)
    train_acc = accuracy_score(y_train, train_preds)
    train_f1m = f1_score(y_train, train_preds, average='macro', zero_division=0)
    cv_f1m = result[f'Mejor {SCORING}']
    gap = train_f1m - cv_f1m

    print(f'{name:<22} {train_acc:>10.4f} {train_f1m:>10.4f} {cv_f1m:>10.4f} {gap:>+10.4f}')